# 階段 3：第一層 —— MA50 月度濾網狀態序列

台股擇時策略研究專案第三階段。

```
第一層（本階段）：MA50 月度濾網  →  決定「可不可以做多」
第二層（階段 5）：RSI 回檔進場     →  決定「什麼時候進」
```

**本階段不做回測、不計算任何績效數字、不使用 RSI 產生任何訊號。**

### 為什麼這一層要單獨檢視

這一層決定策略約 4 成時間空手，是壓低最大回撤的主要機制。
它同時**決定了整個策略的交易次數** —— 後續規則是「持有部位時忽略新訊號、僅由本層決定出場」，
所以每個「可做多」區間最多只會有一次進場。

**本階段產出的 `FLAT → LONG_OK` 切換次數，就是最終策略的交易次數上限。**
這個數字要先看過並確認合理，後面的回測才有意義。

### 沿用階段 1、2 的結論（已確認，不重複檢查）

- `data/twii_indicators.csv`，6,632 筆，1999-01-05 ~ 2026-01-20
- 欄位：Open, High, Low, Close, Volume, MA, RSI3, RSI5
- 指標已驗證正確（RSI 為 Wilder 平滑，經純迴圈交叉驗證）
- MA50 首個有效值 1999-03-24，研究期間可自 2000-01-01 起算
- 研究期間收盤高於 MA50 佔 59.57%

**產出**：`data/regime.csv`

---
## 規則定義

### 狀態判斷

```
在每個月的最後一個交易日 T：
    若 Close[T] > MA50[T]  →  次月狀態 = LONG_OK（可做多）
    否則                     →  次月狀態 = FLAT   （空手）
```

- 比較的是**該月最後一個交易日的收盤價**與**同一天的 MA50 值**
- MA50 是**日線的 50 日簡單移動平均** —— 不是月線的 10 期均線，也不是月均價
- 判斷用的是收盤後才確定的資訊

### 狀態生效時點

```
在月底 T 判斷出的狀態，自「次月第一個交易日的開盤」起生效
```

判斷所需資訊在 T 日收盤已完全確定，行動發生在下一個交易日開盤，**因此沒有 look-ahead**。

實作上最容易錯的地方是展開成日線序列時差一個月：
**某一天的狀態必須來自「上一個月底」的判斷，不是當月月底。**
區塊 4 會用一個獨立的建構方式交叉驗證這件事，不只靠目視。

### 月底的定義

用**資料中實際存在的交易日**決定月底，不用日曆月底 —— 月底常是週末或假日。

---
## 1. 參數與載入

參數集中定義。載入後立刻比對筆數與起訖日期是否與階段 2 一致，不一致就 `raise`。

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ============ 參數 ============
DATA_IN     = "data/twii_indicators.csv"
DATA_OUT    = "data/regime.csv"
MA_LEN      = 50            # 均線濾網天數（與階段 2 一致）
STUDY_START = "2000-01-01"
IS_END      = "2017-12-31"
OOS_START   = "2018-01-01"

# ============ 狀態標籤 ============
LONG_OK = "LONG_OK"
FLAT    = "FLAT"

# ============ 階段 2 已確認的事實 ============
EXPECTED_ROWS  = 6632
EXPECTED_FIRST = "1999-01-05"
EXPECTED_LAST  = "2026-01-20"
EXPECTED_COLS  = ["Open", "High", "Low", "Close", "Volume", "MA", "RSI3", "RSI5"]

# ============ Whipsaw 檢視門檻 ============
SHORT_REGIME_MONTHS = 3      # 持續 3 個月以內視為短區間

pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
plt.rcParams["figure.figsize"] = (14, 7)

# ============ 驗證結果收集器（沿用階段 2 模式） ============
CHECKS = []

def record(name, passed, detail=""):
    verdict = "通過" if passed else "異常"
    CHECKS.append({"驗證項目": name, "結果": verdict, "說明": detail})
    print(f"===> [{verdict}] {name}" + (f"\n      {detail}" if detail else ""))

print("參數設定完成")
print(f"  研究期間 : {STUDY_START} 起")
print(f"  IS       : {STUDY_START} ~ {IS_END}")
print(f"  OOS      : {OOS_START} 起")

參數設定完成
  研究期間 : 2000-01-01 起
  IS       : 2000-01-01 ~ 2017-12-31
  OOS      : 2018-01-01 起


In [2]:
df = pd.read_csv(DATA_IN, index_col="Date", parse_dates=True).sort_index()

load_check = pd.DataFrame({
    "階段 2 確認值": [str(EXPECTED_ROWS), EXPECTED_FIRST, EXPECTED_LAST, ", ".join(EXPECTED_COLS)],
    "本次載入值":    [f"{len(df)}", f"{df.index.min():%Y-%m-%d}",
                      f"{df.index.max():%Y-%m-%d}", ", ".join(df.columns)],
}, index=["筆數", "起始日期", "結束日期", "欄位"])
load_check["一致"] = np.where(
    load_check["階段 2 確認值"] == load_check["本次載入值"], "是", "★ 否")
display(load_check)

if (load_check["一致"] == "★ 否").any():
    raise RuntimeError("載入的資料與階段 2 不一致，請停止並人工確認 data/twii_indicators.csv")

record("資料載入一致性", True,
       f"{len(df):,} 筆，{df.index.min():%Y-%m-%d} ~ {df.index.max():%Y-%m-%d}，"
       f"欄位與階段 2 完全一致")

,階段 2 確認值,本次載入值,一致
筆數,6632,6632,是
起始日期,1999-01-05,1999-01-05,是
結束日期,2026-01-20,2026-01-20,是
欄位,"Open, High, Low, Close, Volume, MA, RSI3, RSI5","Open, High, Low, Close, Volume, MA, RSI3, RSI5",是


===> [通過] 資料載入一致性
      6,632 筆，1999-01-05 ~ 2026-01-20，欄位與階段 2 完全一致


---
## 2. 找出每月最後一個交易日

用資料中**實際存在的交易日**分組取最大值，不用 `pd.date_range` 或日曆月底。

理由很實際：2000-01-31 是星期一有開盤，但 2004-01-31 是星期六。
如果用日曆月底再去對資料，會對到空值或整個月被跳過。
用 `groupby(period).max()` 直接從資料裡取，這個問題就不存在。

同時檢查每個月都恰好有一個月底日（無遺漏、無重複）。

In [3]:
period = df.index.to_period("M")

month_end = df.index.to_series().groupby(period).max()
month_end.index.name = "月份"
month_end.name = "月底交易日"

month_first = df.index.to_series().groupby(period).min()   # 每月第一個交易日
n_days_in_month = df.index.to_series().groupby(period).size()

print(f"資料涵蓋 {len(month_end)} 個月："
      f"{month_end.index.min()} ~ {month_end.index.max()}")
print("\n--- 前 5 個月 ---")
display(month_end.head(5).dt.strftime("%Y-%m-%d").to_frame())
print("--- 後 5 個月 ---")
display(month_end.tail(5).dt.strftime("%Y-%m-%d").to_frame())

資料涵蓋 325 個月：1999-01 ~ 2026-01

--- 前 5 個月 ---


,月底交易日
月份,
1999-01,1999-01-29
1999-02,1999-02-26
1999-03,1999-03-31
1999-04,1999-04-30
1999-05,1999-05-31


--- 後 5 個月 ---


,月底交易日
月份,
2025-09,2025-09-30
2025-10,2025-10-31
2025-11,2025-11-28
2025-12,2025-12-31
2026-01,2026-01-20


In [4]:
# 每個月恰好一個月底日：月份索引不重複，且月份序列連續無跳月
dup_months     = month_end.index.duplicated().sum()
expected_range = pd.period_range(month_end.index.min(), month_end.index.max(), freq="M")
missing_months = expected_range.difference(month_end.index)
# 月底日必須真的落在該月內
wrong_month    = int((month_end.dt.to_period("M") != month_end.index).sum())

me_check = pd.DataFrame({
    "檢查": ["月份索引重複", "月份序列跳月", "月底日不在該月內",
             "月底日總數 vs 月份總數"],
    "值":   [dup_months, len(missing_months), wrong_month,
             f"{len(month_end)} vs {len(expected_range)}"],
})
me_check["結果"] = ["通過" if v in (0, "0") else "★ 異常" for v in me_check["值"][:3]] + \
                   ["通過" if len(month_end) == len(expected_range) else "★ 異常"]
display(me_check)

me_ok = (me_check["結果"] == "通過").all()
record("每月月底交易日唯一且完整", me_ok,
       f"{len(month_end)} 個月，每月恰好一個月底交易日，無遺漏、無重複、無跳月"
       if me_ok else "月底交易日的擷取有問題，明細見上表")

,檢查,值,結果
0,月份索引重複,0,通過
1,月份序列跳月,0,通過
2,月底日不在該月內,0,通過
3,月底日總數 vs 月份總數,325 vs 325,通過


===> [通過] 每月月底交易日唯一且完整
      325 個月，每月恰好一個月底交易日，無遺漏、無重複、無跳月


---
## 3. 月度狀態判斷

對每個月底 T 比較 `Close[T]` 與 `MA50[T]`，決定**次月**的狀態。

「生效起始日」是次月第一個交易日。最後一個月（2026-01）沒有次月，生效起始日留空 ——
這代表 2026-01 月底判斷出的狀態要到 2026-02 才生效，超出目前資料範圍，屬正常。

1999-10 之前 MA50 尚未有值，狀態為未定義（`NaN`），不影響研究期間
（研究期間第一個月 2000-01 的狀態由 1999-12 月底決定，該日 MA50 早已有效）。

In [5]:
me = df.loc[month_end.values, ["Close", "MA"]].copy()
me.index = month_end.index                       # 以「判斷所屬月份」為索引
me.insert(0, "月底交易日", month_end.values)

# 狀態判斷：收盤 > MA50 -> 次月可做多
signal = me["Close"] > me["MA"]
me["判斷狀態"] = np.where(signal, LONG_OK, FLAT)
me.loc[me["MA"].isna(), "判斷狀態"] = np.nan   # MA50 未成形前不判斷

me["收盤/MA50"] = me["Close"] / me["MA"]

# 生效起始日 = 次月第一個交易日
next_first = month_first.reindex(me.index + 1)
me["生效起始日"] = next_first.values

monthly = me[["月底交易日", "Close", "MA", "收盤/MA50", "判斷狀態", "生效起始日"]].copy()
monthly.columns = ["月底交易日", "收盤價", f"MA{MA_LEN}", "收盤/MA50", "判斷狀態", "生效起始日"]

def _fmt(frame):
    out = frame.copy()
    for c in ["月底交易日", "生效起始日"]:
        out[c] = pd.to_datetime(out[c]).dt.strftime("%Y-%m-%d").fillna("—（無次月）")
    return out

print("--- 前 12 個月 ---")
display(_fmt(monthly.head(12)))
print("--- 後 12 個月 ---")
display(_fmt(monthly.tail(12)))

--- 前 12 個月 ---


,月底交易日,收盤價,MA50,收盤/MA50,判斷狀態,生效起始日
月份,,,,,,
1999-01,1999-01-29,"5,984.00",NaN,NaN,NaN,1999-02-01
1999-02,1999-02-26,"6,318.52",NaN,NaN,NaN,1999-03-01
1999-03,1999-03-31,"6,881.72","6,356.92",1.08,LONG_OK,1999-04-01
1999-04,1999-04-30,"7,371.17","6,917.01",1.07,LONG_OK,1999-05-03
1999-05,1999-05-31,"7,316.57","7,366.94",0.99,FLAT,1999-06-01
1999-06,1999-06-30,"8,467.37","7,742.10",1.09,LONG_OK,1999-07-02
1999-07,1999-07-30,"7,413.11","7,922.43",0.94,FLAT,1999-08-02
1999-08,1999-08-31,"8,157.73","7,889.62",1.03,LONG_OK,1999-09-01
1999-09,1999-09-30,"7,598.79","7,752.06",0.98,FLAT,1999-10-01


--- 後 12 個月 ---


,月底交易日,收盤價,MA50,收盤/MA50,判斷狀態,生效起始日
月份,,,,,,
2025-02,2025-02-27,"23,053.18","23,173.73",0.99,FLAT,2025-03-03
2025-03,2025-03-31,"20,695.90","22,789.30",0.91,FLAT,2025-04-01
2025-04,2025-04-30,"20,235.03","21,370.78",0.95,FLAT,2025-05-02
2025-05,2025-05-29,"21,347.30","20,725.71",1.03,LONG_OK,2025-06-02
2025-06,2025-06-30,"22,256.02","21,290.01",1.05,LONG_OK,2025-07-01
2025-07,2025-07-31,"23,542.52","22,386.86",1.05,LONG_OK,2025-08-01
2025-08,2025-08-29,"24,233.10","23,335.42",1.04,LONG_OK,2025-09-01
2025-09,2025-09-30,"25,820.54","24,425.54",1.06,LONG_OK,2025-10-01
2025-10,2025-10-31,"28,233.35","25,899.43",1.09,LONG_OK,2025-11-03


In [6]:
undecided = monthly["判斷狀態"].isna().sum()
first_decided = monthly["判斷狀態"].first_valid_index()

state_counts = monthly["判斷狀態"].value_counts(dropna=False).to_frame("月數")
state_counts.index.name = "判斷狀態"
display(state_counts)

record("月度狀態判斷完成", True,
       f"共 {len(monthly)} 個月，其中 {undecided} 個月因 MA{MA_LEN} 未成形而未判斷"
       f"（{monthly.index.min()} ~ {monthly.index[monthly.index.get_loc(first_decided) - 1]}），"
       f"首個可判斷月份為 {first_decided}，決定 {first_decided + 1} 的狀態")

,月數
判斷狀態,
LONG_OK,199
FLAT,124
NaN,2


===> [通過] 月度狀態判斷完成
      共 325 個月，其中 2 個月因 MA50 未成形而未判斷（1999-01 ~ 1999-02），首個可判斷月份為 1999-03，決定 1999-04 的狀態


---
## 4. 展開為日線狀態序列 ★ 最容易出錯的地方

規則是「某一天的狀態來自**上一個月底**的判斷」。
展開時只要差一個月，就變成用當月月底的資訊決定當月的行動 —— 這是典型的 look-ahead，
而且程式不會報任何錯，回測績效會憑空變好。

**主要建構方式**：把月份索引 `+1` 之後對齊回日線，等價於「本月的狀態查上個月的判斷」。

In [7]:
# 主要建構：狀態表的索引 +1 = 該狀態生效的月份
state_by_effective_month = monthly["判斷狀態"].copy()
state_by_effective_month.index = state_by_effective_month.index + 1

regime = pd.Series(
    state_by_effective_month.reindex(df.index.to_period("M")).values,
    index=df.index, name="regime")

print(regime.loc[STUDY_START:].value_counts().to_string())
print(f"\n研究期間首日 {regime.loc[STUDY_START:].index[0]:%Y-%m-%d} 的狀態："
      f"{regime.loc[STUDY_START:].iloc[0]}")

regime
LONG_OK    3908
FLAT       2483

研究期間首日 2000-01-04 的狀態：LONG_OK


### 4a. 獨立建構交叉驗證

上面用的是「月份週期 +1」的算法。這裡改用**完全不同的路徑**重建一次：
以每個月度判斷的「生效起始日」為事件點，在日線上向前填補（forward fill）。

兩條路徑若在全序列上完全一致，才能確認展開沒有錯位。
只用同一套邏輯再寫一遍不算驗證 —— 那只會把同樣的錯誤重現一次。

In [8]:
# 獨立路徑：以「生效起始日」為事件，ffill 到後續每個交易日
events = monthly.dropna(subset=["生效起始日"])[["生效起始日", "判斷狀態"]]
alt = pd.Series(np.nan, index=df.index, dtype=object)
alt.loc[pd.to_datetime(events["生效起始日"]).values] = events["判斷狀態"].values
alt = alt.ffill()

both_na = regime.isna() & alt.isna()
identical = bool(((regime == alt) | both_na).all())

mismatch = df.index[~((regime == alt) | both_na)]
if len(mismatch):
    display(pd.DataFrame({"主要建構": regime.loc[mismatch],
                          "獨立建構": alt.loc[mismatch]}).head(20))

record("日線狀態序列：兩種獨立建構一致", identical,
       f"「月份 +1 對齊」與「生效日 ffill」兩條路徑在全部 {len(df):,} 個交易日上完全一致"
       if identical else f"兩種建構有 {len(mismatch)} 天不一致，明細見上表")

===> [通過] 日線狀態序列：兩種獨立建構一致
      「月份 +1 對齊」與「生效日 ffill」兩條路徑在全部 6,632 個交易日上完全一致


### 4b. 程式化斷言：狀態必須來自上一個月底

對研究期間的**每一個交易日**逐日檢查：
該日的 `regime`，是否等於「該日所屬月份的前一個月，其最後一個交易日」的判斷結果。

這是把規則直接翻譯成斷言，不依賴前面任何一種建構方式。

In [9]:
chk = df.loc[STUDY_START:].copy()
cur_period  = chk.index.to_period("M")
prev_period = cur_period - 1

# 該日所屬月份的「上一個月底交易日」
prev_me_date = month_end.reindex(prev_period).values
prev_close   = df["Close"].reindex(prev_me_date).values
prev_ma      = df["MA"].reindex(prev_me_date).values
expected     = np.where(prev_close > prev_ma, LONG_OK, FLAT)

actual = regime.loc[STUDY_START:].values
bad = np.where(actual != expected)[0]

assert_ok = len(bad) == 0
if not assert_ok:
    display(pd.DataFrame({
        "日期": chk.index[bad].strftime("%Y-%m-%d"),
        "上月底": pd.to_datetime(prev_me_date[bad]).strftime("%Y-%m-%d"),
        "應為": expected[bad], "實際": actual[bad],
    }).head(20))

# 順帶確認：用來判斷的月底，確實落在該日的前一個月
month_gap_ok = bool((pd.DatetimeIndex(prev_me_date).to_period("M") == prev_period).all())

record("狀態來自上一個月底（程式化斷言）", assert_ok and month_gap_ok,
       f"研究期間 {len(chk):,} 個交易日全部通過：每日 regime 皆等於其前一月份最後一個"
       f"交易日的 Close vs MA{MA_LEN} 判斷結果，且判斷日確實落在前一個月"
       if assert_ok and month_gap_ok else f"有 {len(bad)} 天不符合規則，明細見上表")

===> [通過] 狀態來自上一個月底（程式化斷言）
      研究期間 6,391 個交易日全部通過：每日 regime 皆等於其前一月份最後一個交易日的 Close vs MA50 判斷結果，且判斷日確實落在前一個月


### 4c. 目視檢查：抽 5 個切換點看前後 3 天

固定亂數種子抽 5 個狀態切換點，逐筆印出切換前後各 3 個交易日。

要看的是：**狀態確實在次月第一個交易日才改變**，
而且改變的依據是上一列（上個月最後一天）的 Close 與 MA50 關係。

In [10]:
r_study = regime.loc[STUDY_START:]
switch_dates = r_study.index[(r_study != r_study.shift(1)) & r_study.shift(1).notna()]

rng_ = np.random.default_rng(42)
picked = sorted(rng_.choice(len(switch_dates), size=5, replace=False))
sample = [switch_dates[i] for i in picked]   # 取位置再索引，確保拿到 Timestamp

for d in sample:
    i = df.index.get_loc(d)
    win = df.iloc[max(i - 3, 0): i + 3][["Close", "MA"]].copy()
    win["regime"] = regime.loc[win.index]
    win["是該月最後交易日"] = np.where(win.index.isin(month_end.values), "★ 是", "")
    win["切換日"] = np.where(win.index == d, "◀ 切換", "")
    win.index = win.index.strftime("%Y-%m-%d (%a)")
    print(f"\n=== 切換點 {d:%Y-%m-%d}："
          f"{regime.loc[:d].iloc[-2]} → {regime.loc[d]} ===")
    display(win)


=== 切換點 2003-03-03：LONG_OK → FLAT ===


,Close,MA,regime,是該月最後交易日,切換日
Date,,,,,
2003-02-25 (Tue),"4,454.35","4,707.27",LONG_OK,,
2003-02-26 (Wed),"4,456.69","4,699.93",LONG_OK,,
2003-02-27 (Thu),"4,432.46","4,693.48",LONG_OK,★ 是,
2003-03-03 (Mon),"4,526.69","4,690.03",FLAT,,◀ 切換
2003-03-04 (Tue),"4,499.69","4,686.63",FLAT,,
2003-03-05 (Wed),"4,418.11","4,683.23",FLAT,,



=== 切換點 2011-12-01：LONG_OK → FLAT ===


,Close,MA,regime,是該月最後交易日,切換日
Date,,,,,
2011-11-28 (Mon),"6,898.78","7,309.04",LONG_OK,,
2011-11-29 (Tue),"6,988.65","7,299.19",LONG_OK,,
2011-11-30 (Wed),"6,904.12","7,287.42",LONG_OK,★ 是,
2011-12-01 (Thu),"7,178.69","7,280.27",FLAT,,◀ 切換
2011-12-02 (Fri),"7,140.68","7,276.98",FLAT,,
2011-12-05 (Mon),"7,098.08","7,278.01",FLAT,,



=== 切換點 2017-12-01：LONG_OK → FLAT ===


,Close,MA,regime,是該月最後交易日,切換日
Date,,,,,
2017-11-28 (Tue),"10,707.07","10,659.06",LONG_OK,,
2017-11-29 (Wed),"10,713.55","10,661.73",LONG_OK,,
2017-11-30 (Thu),"10,560.44","10,660.30",LONG_OK,★ 是,
2017-12-01 (Fri),"10,600.37","10,660.79",FLAT,,◀ 切換
2017-12-04 (Mon),"10,651.11","10,663.43",FLAT,,
2017-12-05 (Tue),"10,566.85","10,663.20",FLAT,,



=== 切換點 2019-10-01：FLAT → LONG_OK ===


,Close,MA,regime,是該月最後交易日,切換日
Date,,,,,
2019-09-25 (Wed),"10,873.69","10,686.27",FLAT,,
2019-09-26 (Thu),"10,871.99","10,686.18",FLAT,,
2019-09-27 (Fri),"10,829.68","10,685.05",FLAT,★ 是,
2019-10-01 (Tue),"10,967.65","10,687.83",LONG_OK,,◀ 切換
2019-10-02 (Wed),"10,947.88","10,690.81",LONG_OK,,
2019-10-03 (Thu),"10,875.91","10,690.86",LONG_OK,,



=== 切換點 2025-06-02：FLAT → LONG_OK ===


,Close,MA,regime,是該月最後交易日,切換日
Date,,,,,
2025-05-27 (Tue),"21,336.54","20,753.34",FLAT,,
2025-05-28 (Wed),"21,357.72","20,741.13",FLAT,,
2025-05-29 (Thu),"21,347.30","20,725.71",FLAT,★ 是,
2025-06-02 (Mon),"21,002.71","20,700.33",LONG_OK,,◀ 切換
2025-06-03 (Tue),"21,126.93","20,683.65",LONG_OK,,
2025-06-04 (Wed),"21,618.09","20,668.47",LONG_OK,,


In [11]:
# 切換是否只發生在每月第一個交易日
first_days = set(month_first.values)
bad_switch = [d for d in switch_dates if np.datetime64(d) not in
              set(np.datetime64(x) for x in first_days)]

na_after_start = int(regime.loc[STUDY_START:].isna().sum())

vis = pd.DataFrame({
    "檢查": ["狀態切換日皆為當月第一個交易日",
             f"{STUDY_START} 之後 regime 無缺值"],
    "違反數": [len(bad_switch), na_after_start],
})
vis["結果"] = np.where(vis["違反數"] == 0, "通過", "★ 異常")
display(vis)

ok = (vis["違反數"] == 0).all()
record("切換時點與缺值檢查", ok,
       f"全部 {len(switch_dates)} 次切換皆發生在當月第一個交易日；"
       f"{STUDY_START} 之後 {len(regime.loc[STUDY_START:]):,} 個交易日無任何 regime 缺值"
       if ok else "切換時點或缺值有問題，明細見上表")

,檢查,違反數,結果
0,狀態切換日皆為當月第一個交易日,0,通過
1,2000-01-01 之後 regime 無缺值,0,通過


===> [通過] 切換時點與缺值檢查
      全部 98 次切換皆發生在當月第一個交易日；2000-01-01 之後 6,391 個交易日無任何 regime 缺值


---
## 5. 狀態切換完整清單 ★ 本階段核心產出

研究期間內每一次狀態切換，完整列出不截斷。

`FLAT → LONG_OK` 的次數就是最終策略的**交易次數上限** ——
因為後續規則是「持有部位時忽略新訊號、僅由本層決定出場」，
每個 `LONG_OK` 區間最多只會有一次進場。

「前一狀態持續月數／交易日數」指的是**切換發生之前**那一段區間的長度。
研究期間的第一段（2000-01 起算）並非完整區間，因此第一列的持續長度以 `—` 表示。

In [12]:
def regime_intervals(series):
    """把狀態序列切成連續區間，回傳 (起日, 迄日, 狀態, 交易日數, 月數)。"""
    s = series.dropna()
    grp = (s != s.shift(1)).cumsum()
    out = []
    for _, seg in s.groupby(grp):
        out.append({
            "起始日": seg.index[0], "結束日": seg.index[-1], "狀態": seg.iloc[0],
            "交易日數": len(seg),
            "月數": len(seg.index.to_period("M").unique()),
        })
    return pd.DataFrame(out)

intervals = regime_intervals(r_study)
print(f"研究期間共切成 {len(intervals)} 段區間")
display(intervals.head(3).assign(
    起始日=lambda x: x["起始日"].dt.strftime("%Y-%m-%d"),
    結束日=lambda x: x["結束日"].dt.strftime("%Y-%m-%d")))

研究期間共切成 99 段區間


,起始日,結束日,狀態,交易日數,月數
0,2000-01-04,2000-04-28,LONG_OK,76,4
1,2000-05-02,2001-01-31,FLAT,185,9
2,2001-02-01,2001-04-30,LONG_OK,61,3


In [13]:
rows = []
for k in range(1, len(intervals)):
    prev, cur = intervals.iloc[k - 1], intervals.iloc[k]
    d = cur["起始日"]
    rows.append({
        "#": k,
        "切換日期": f"{d:%Y-%m-%d}",
        "切換方向": f"{prev['狀態']} → {cur['狀態']}",
        "當日開盤": df.loc[d, "Open"],
        "當日收盤": df.loc[d, "Close"],
        "前一狀態持續月數": prev["月數"] if k > 1 else np.nan,
        "前一狀態持續交易日數": prev["交易日數"] if k > 1 else np.nan,
    })

switches = pd.DataFrame(rows).set_index("#")
switches_disp = switches.copy()
for c in ["前一狀態持續月數", "前一狀態持續交易日數"]:
    switches_disp[c] = switches_disp[c].map(lambda v: "—" if pd.isna(v) else f"{int(v)}")

print(f"研究期間狀態切換共 {len(switches)} 次"
      f"（{(switches['切換方向'] == f'{FLAT} → {LONG_OK}').sum()} 次進場方向，"
      f"{(switches['切換方向'] == f'{LONG_OK} → {FLAT}').sum()} 次出場方向）")
display(switches_disp)

研究期間狀態切換共 98 次（49 次進場方向，49 次出場方向）


,切換日期,切換方向,當日開盤,當日收盤,前一狀態持續月數,前一狀態持續交易日數
#,,,,,,
1,2000-05-02,LONG_OK → FLAT,"8,836.83","8,638.75",—,—
2,2001-02-01,FLAT → LONG_OK,"5,927.25","5,897.93",9,185
3,2001-05-02,LONG_OK → FLAT,"5,469.95","5,304.24",3,61
4,2001-12-03,FLAT → LONG_OK,"4,534.37","4,646.61",7,147
5,2002-05-02,LONG_OK → FLAT,"6,099.27","5,867.83",5,97
6,2002-11-01,FLAT → LONG_OK,"4,596.69","4,500.55",6,129
7,2003-01-02,LONG_OK → FLAT,"4,460.57","4,524.87",2,43
8,2003-02-06,FLAT → LONG_OK,"4,975.65","4,833.58",1,19
9,2003-03-03,LONG_OK → FLAT,"4,483.44","4,526.69",1,16


In [14]:
n_entry = int((switches["切換方向"] == f"{FLAT} → {LONG_OK}").sum())
n_exit  = int((switches["切換方向"] == f"{LONG_OK} → {FLAT}").sum())

record("狀態切換清單", True,
       f"研究期間共 {len(switches)} 次切換：{n_entry} 次 {FLAT}→{LONG_OK}、"
       f"{n_exit} 次 {LONG_OK}→{FLAT}。"
       f"其中 {n_entry} 即為最終策略的交易次數上限（僅記錄，不作判斷）")

===> [通過] 狀態切換清單
      研究期間共 98 次切換：49 次 FLAT→LONG_OK、49 次 LONG_OK→FLAT。其中 49 即為最終策略的交易次數上限（僅記錄，不作判斷）


---
## 6. 切換時點與重大事件對照

六段已知重大下跌，各回答兩個問題：

1. **退場訊號發生在哪一天？** 當時距該波高點跌了多少 %
2. **重新進場在哪一天？** 當時距該波低點漲了多少 %

### 區間定義

波段高低點用**寫死的檢視區間內的實際收盤極值**，不做任何自動偵測 ——
自動偵測會引入額外的參數與判斷，反而讓這張表難以解釋。
區間是依已知事件的日期範圍手動設定，寬鬆到足以涵蓋該波的高低點：

| 事件 | 找高點的區間 | 找低點的區間 |
|---|---|---|
| 網路泡沫 | 2000-01-01 ~ 2000-04-30 | 2001-06-01 ~ 2001-12-31 |
| 金融海嘯 | 2007-07-01 ~ 2008-06-30 | 2008-09-01 ~ 2009-03-31 |
| 歐債危機 | 2011-01-01 ~ 2011-08-31 | 2011-09-01 ~ 2012-01-31 |
| 中國股災 | 2015-01-01 ~ 2015-05-31 | 2015-08-01 ~ 2016-02-29 |
| COVID | 2020-01-01 ~ 2020-02-29 | 2020-03-01 ~ 2020-04-30 |
| 升息循環 | 2021-11-01 ~ 2022-04-30 | 2022-09-01 ~ 2023-01-31 |

「退場」取該波高點日之後**第一次** `LONG_OK → FLAT`。

「重新進場」列兩個版本，因為兩者不一定相同：

- **退場後首次** —— 退場日之後第一次 `FLAT → LONG_OK`。這是規則實際做的事。
- **低點後首次** —— 波段低點之後第一次 `FLAT → LONG_OK`。

若下跌途中出現 whipsaw（短暫回到均線上方又跌破），
「退場後首次」會落在波段低點**之前**，此時它與低點的價差不代表「從低點漲了多少」。
兩欄並列可以直接看出哪幾段發生了這種情況，補充表的「兩者相同」欄會標示出來。

點位一律用**該切換日的開盤價** —— 規則是次月第一個交易日開盤生效，
開盤價才是實際可成交的價格。高低點則用收盤價。

**這張表只呈現數字，不下判斷、不建議調整參數。**

In [15]:
EVENTS = [
    ("Dot-com bust",      ("2000-01-01", "2000-04-30"), ("2001-06-01", "2001-12-31"),
     ("1999-06-01", "2002-06-30")),
    ("Global fin. crisis", ("2007-07-01", "2008-06-30"), ("2008-09-01", "2009-03-31"),
     ("2007-01-01", "2010-06-30")),
    ("Euro debt crisis",  ("2011-01-01", "2011-08-31"), ("2011-09-01", "2012-01-31"),
     ("2010-06-01", "2012-12-31")),
    ("China selloff 2015", ("2015-01-01", "2015-05-31"), ("2015-08-01", "2016-02-29"),
     ("2014-06-01", "2016-12-31")),
    ("COVID-19",          ("2020-01-01", "2020-02-29"), ("2020-03-01", "2020-04-30"),
     ("2019-06-01", "2021-06-30")),
    ("Rate-hike 2022",    ("2021-11-01", "2022-04-30"), ("2022-09-01", "2023-01-31"),
     ("2021-01-01", "2023-12-31")),
]

def first_switch(after, direction):
    """回傳 after 之後第一次指定方向的切換日期，沒有則 None。"""
    cand = switches[(pd.to_datetime(switches["切換日期"]) >= pd.Timestamp(after))
                    & (switches["切換方向"] == direction)]
    if cand.empty:
        return None
    return pd.Timestamp(cand.iloc[0]["切換日期"])

rows = []
for name, hi_win, lo_win, _ in EVENTS:
    hi_seg = df.loc[hi_win[0]:hi_win[1], "Close"]
    lo_seg = df.loc[lo_win[0]:lo_win[1], "Close"]
    hi_d, hi_p = hi_seg.idxmax(), hi_seg.max()
    lo_d, lo_p = lo_seg.idxmin(), lo_seg.min()

    ex_d = first_switch(hi_d, f"{LONG_OK} → {FLAT}")
    # 兩種「重新進場」：退場後第一次，以及波段低點後第一次
    en_d  = first_switch(ex_d, f"{FLAT} → {LONG_OK}") if ex_d is not None else None
    en2_d = first_switch(lo_d, f"{FLAT} → {LONG_OK}")

    ex_p  = df.loc[ex_d, "Open"]  if ex_d  is not None else np.nan
    en_p  = df.loc[en_d, "Open"]  if en_d  is not None else np.nan
    en2_p = df.loc[en2_d, "Open"] if en2_d is not None else np.nan

    rows.append({
        "事件": name,
        "波段高點日": f"{hi_d:%Y-%m-%d}", "高點收盤": hi_p,
        "退場日": f"{ex_d:%Y-%m-%d}" if ex_d is not None else "—",
        "退場開盤": ex_p,
        "高點→退場": (ex_p / hi_p - 1) if ex_d is not None else np.nan,
        "波段低點日": f"{lo_d:%Y-%m-%d}", "低點收盤": lo_p,
        "進場日（退場後首次）": f"{en_d:%Y-%m-%d}" if en_d is not None else "—",
        "進場開盤": en_p,
        "低點→進場": (en_p / lo_p - 1) if en_d is not None else np.nan,
        "進場日（低點後首次）": f"{en2_d:%Y-%m-%d}" if en2_d is not None else "—",
        "進場開盤2": en2_p,
        "低點→進場2": (en2_p / lo_p - 1) if en2_d is not None else np.nan,
    })

events_tbl = pd.DataFrame(rows).set_index("事件")

MAIN_COLS = ["波段高點日", "高點收盤", "退場日", "退場開盤", "高點→退場",
             "波段低點日", "低點收盤", "進場日（退場後首次）", "進場開盤", "低點→進場"]
print("--- 主表：退場後第一次重新進場 ---")
display(events_tbl[MAIN_COLS].style.format({
    "高點收盤": "{:,.0f}", "退場開盤": "{:,.0f}", "低點收盤": "{:,.0f}",
    "進場開盤": "{:,.0f}", "高點→退場": "{:+.1%}", "低點→進場": "{:+.1%}"}))

--- 主表：退場後第一次重新進場 ---

,波段高點日,高點收盤,退場日,退場開盤,高點→退場,波段低點日,低點收盤,進場日（退場後首次）,進場開盤,低點→進場
事件,,,,,,,,,,
Dot-com bust,2000-02-17,"10,202",2000-05-02,"8,837",-13.4%,2001-10-03,"3,446",2001-02-01,"5,927",+72.0%
Global fin. crisis,2007-10-29,"9,810",2007-12-03,"8,623",-12.1%,2008-11-20,"4,090",2008-03-03,"8,214",+100.8%
Euro debt crisis,2011-01-28,"9,145",2011-03-01,"8,612",-5.8%,2011-12-19,"6,633",2011-05-03,"9,014",+35.9%
China selloff 2015,2015-04-27,"9,973",2015-07-01,"9,312",-6.6%,2015-08-24,"7,410",2015-11-02,"8,572",+15.7%
COVID-19,2020-01-14,"12,180",2020-02-03,"11,366",-6.7%,2020-03-19,"8,681",2020-05-04,"10,782",+24.2%
Rate-hike 2022,2022-01-04,"18,526",2022-02-07,"17,751",-4.2%,2022-10-25,"12,666",2022-06-01,"16,719",+32.0%


In [16]:
# 補充表：波段低點之後第一次進場。兩者不同時，代表中間出現了 whipsaw 進出
supp = events_tbl[["波段低點日", "低點收盤",
                   "進場日（退場後首次）", "低點→進場",
                   "進場日（低點後首次）", "進場開盤2", "低點→進場2"]].copy()
supp["兩者相同"] = np.where(
    supp["進場日（退場後首次）"] == supp["進場日（低點後首次）"], "是", "★ 否")
print("--- 補充表：低點後第一次重新進場 ---")
display(supp.style.format({
    "低點收盤": "{:,.0f}", "進場開盤2": "{:,.0f}",
    "低點→進場": "{:+.1%}", "低點→進場2": "{:+.1%}"}))

--- 補充表：低點後第一次重新進場 ---


,波段低點日,低點收盤,進場日（退場後首次）,低點→進場,進場日（低點後首次）,進場開盤2,低點→進場2,兩者相同
事件,,,,,,,,
Dot-com bust,2001-10-03,"3,446",2001-02-01,+72.0%,2001-12-03,"4,534",+31.6%,★ 否
Global fin. crisis,2008-11-20,"4,090",2008-03-03,+100.8%,2009-01-05,"4,725",+15.5%,★ 否
Euro debt crisis,2011-12-19,"6,633",2011-05-03,+35.9%,2012-02-01,"7,520",+13.4%,★ 否
China selloff 2015,2015-08-24,"7,410",2015-11-02,+15.7%,2015-11-02,"8,572",+15.7%,是
COVID-19,2020-03-19,"8,681",2020-05-04,+24.2%,2020-05-04,"10,782",+24.2%,是
Rate-hike 2022,2022-10-25,"12,666",2022-06-01,+32.0%,2022-12-01,"15,060",+18.9%,★ 否


In [17]:
# 補充：退場與進場相對高低點各晚了幾個交易日
lag = []
for (name, hi_win, lo_win, _), (_, r) in zip(EVENTS, events_tbl.iterrows()):
    hi_d = pd.Timestamp(r["波段高點日"]); lo_d = pd.Timestamp(r["波段低點日"])
    ex_d  = pd.Timestamp(r["退場日"]) if r["退場日"] != "—" else None
    en_d  = pd.Timestamp(r["進場日（退場後首次）"]) if r["進場日（退場後首次）"] != "—" else None
    en2_d = pd.Timestamp(r["進場日（低點後首次）"]) if r["進場日（低點後首次）"] != "—" else None
    loc = df.index.get_loc
    lag.append({
        "事件": name,
        "高點→退場（交易日）": loc(ex_d) - loc(hi_d) if ex_d is not None else np.nan,
        "退場→進場（交易日）": loc(en_d) - loc(ex_d) if (ex_d is not None and en_d is not None) else np.nan,
        "低點→進場・退場後首次": loc(en_d) - loc(lo_d) if en_d is not None else np.nan,
        "低點→進場・低點後首次": loc(en2_d) - loc(lo_d) if en2_d is not None else np.nan,
    })
lag_tbl = pd.DataFrame(lag).set_index("事件")
display(lag_tbl)
print("「低點→進場・退場後首次」為負值，代表該次進場發生在波段低點之前（下跌途中的 whipsaw）。")

n_whip = int((events_tbl["進場日（退場後首次）"] != events_tbl["進場日（低點後首次）"]).sum())
record("重大事件切換時點對照", True,
       f"六段重大下跌的退場／進場時點已完整列表（高點→退場跌幅 "
       f"{events_tbl['高點→退場'].min():+.1%} ~ {events_tbl['高點→退場'].max():+.1%}）；"
       f"其中 {n_whip} 段的「退場後首次進場」早於波段低點（下跌途中的 whipsaw）。"
       "僅記錄事實，不作判斷")

,高點→退場（交易日）,退場→進場（交易日）,低點→進場・退場後首次,低點→進場・低點後首次
事件,,,,
Dot-com bust,49,185,-166,42
Global fin. crisis,25,57,-183,30
Euro debt crisis,15,42,-161,24
China selloff 2015,45,84,47,47
COVID-19,7,61,29,29
Rate-hike 2022,17,78,-101,27


「低點→進場・退場後首次」為負值，代表該次進場發生在波段低點之前（下跌途中的 whipsaw）。
===> [通過] 重大事件切換時點對照
      六段重大下跌的退場／進場時點已完整列表（高點→退場跌幅 -13.4% ~ -4.2%）；其中 4 段的「退場後首次進場」早於波段低點（下跌途中的 whipsaw）。僅記錄事實，不作判斷


---
## 7. 狀態統計

四組數字：整體佔比、IS/OOS 分組佔比、切換次數、區間長度分布。

In [18]:
def state_share(seg, label):
    vc = seg.value_counts()
    total = len(seg)
    out = pd.DataFrame({
        "交易日數": [vc.get(LONG_OK, 0), vc.get(FLAT, 0), total],
        "比例": [f"{vc.get(LONG_OK, 0) / total:.2%}",
                 f"{vc.get(FLAT, 0) / total:.2%}", "100.00%"],
    }, index=[LONG_OK, FLAT, "合計"])
    out.columns = pd.MultiIndex.from_product([[label], out.columns])
    return out

share = pd.concat([
    state_share(r_study, f"研究期間（{STUDY_START} ~）"),
    state_share(regime.loc[STUDY_START:IS_END], f"IS（~ {IS_END}）"),
    state_share(regime.loc[OOS_START:], f"OOS（{OOS_START} ~）"),
], axis=1)
share.index.name = "狀態"
display(share)

研究期間（2000-01-01 ~）          IS（~ 2017-12-31）          OOS（2018-01-01 ~）         
                      交易日數       比例             交易日數       比例              交易日數       比例
狀態                                                                                      
LONG_OK               3908   61.15%             2651   59.79%              1257   64.23%
FLAT                  2483   38.85%             1783   40.21%               700   35.77%
合計                    6391  100.00%             4434  100.00%              1957  100.00%

In [19]:
sw_dt = pd.to_datetime(switches["切換日期"])
cnt = pd.DataFrame({
    "全期間": [len(switches), n_entry, n_exit],
    f"IS（~{IS_END}）": [
        int((sw_dt <= IS_END).sum()),
        int(((sw_dt <= IS_END) & (switches["切換方向"] == f"{FLAT} → {LONG_OK}")).sum()),
        int(((sw_dt <= IS_END) & (switches["切換方向"] == f"{LONG_OK} → {FLAT}")).sum()),
    ],
    f"OOS（{OOS_START}~）": [
        int((sw_dt >= OOS_START).sum()),
        int(((sw_dt >= OOS_START) & (switches["切換方向"] == f"{FLAT} → {LONG_OK}")).sum()),
        int(((sw_dt >= OOS_START) & (switches["切換方向"] == f"{LONG_OK} → {FLAT}")).sum()),
    ],
}, index=["切換總次數", f"{FLAT} → {LONG_OK}（＝交易次數上限）", f"{LONG_OK} → {FLAT}"])
display(cnt)

,全期間,IS（~2017-12-31）,OOS（2018-01-01~）
切換總次數,98,63,35
FLAT → LONG_OK（＝交易次數上限）,49,31,18
LONG_OK → FLAT,49,32,17


In [20]:
by_year = pd.DataFrame({
    f"{LONG_OK} 天數": r_study.groupby(r_study.index.year).apply(lambda s: int((s == LONG_OK).sum())),
    "交易日數":        r_study.groupby(r_study.index.year).size(),
    "切換次數":        sw_dt.groupby(sw_dt.dt.year).size(),
}).fillna(0).astype(int)
by_year[f"{LONG_OK} 比例"] = (by_year[f"{LONG_OK} 天數"] / by_year["交易日數"]).map("{:.0%}".format)
by_year.index.name = "年份"
display(by_year)

,LONG_OK 天數,交易日數,切換次數,LONG_OK 比例
年份,,,,
2000,76,245,1,31%
2001,82,245,3,33%
2002,119,248,2,48%
2003,143,249,5,57%
2004,84,250,4,34%
2005,143,247,5,58%
2006,181,247,2,73%
2007,205,243,3,84%
2008,63,249,2,25%


In [21]:
length_stats = intervals.groupby("狀態")["交易日數"].agg(
    段數="count", 最短="min", 中位數="median", 平均="mean", 最長="max")
length_stats_m = intervals.groupby("狀態")["月數"].agg(
    最短月="min", 中位數月="median", 平均月="mean", 最長月="max")
display(pd.concat([length_stats, length_stats_m], axis=1).round(1))

record("狀態統計", True,
       f"研究期間 {LONG_OK} {(r_study == LONG_OK).mean():.2%}／"
       f"{FLAT} {(r_study == FLAT).mean():.2%}；"
       f"共 {len(intervals)} 段區間，{LONG_OK} 區間中位數 "
       f"{intervals[intervals['狀態'] == LONG_OK]['交易日數'].median():.0f} 交易日、"
       f"{FLAT} 區間中位數 "
       f"{intervals[intervals['狀態'] == FLAT]['交易日數'].median():.0f} 交易日")

,段數,最短,中位數,平均,最長,最短月,中位數月,平均月,最長月
狀態,,,,,,,,,
FLAT,49,17,42.00,50.70,185,1,2.00,2.40,9
LONG_OK,50,13,59.00,78.20,327,1,3.00,3.90,16


===> [通過] 狀態統計
      研究期間 LONG_OK 61.15%／FLAT 38.85%；共 99 段區間，LONG_OK 區間中位數 59 交易日、FLAT 區間中位數 42 交易日


---
## 8. Whipsaw 檢視

列出所有**持續 3 個月以內**的狀態區間，無論方向。這些是來回巴的候選。

月度判斷天生就有這個問題：指數在 MA50 附近盤整時，
月底收盤在均線上下反覆，狀態就會跟著月月切換。

**這一格只呈現數字，不下判斷、不做任何處理。**
是否需要加確認天數或緩衝區，由人決定。

In [22]:
short = intervals[intervals["月數"] <= SHORT_REGIME_MONTHS].copy()

summary_short = pd.DataFrame({
    "值": [
        f"{len(intervals)}",
        f"{len(short)}",
        f"{len(short) / len(intervals):.1%}",
        f"{int((short['狀態'] == LONG_OK).sum())}",
        f"{int((short['狀態'] == FLAT).sum())}",
        f"{int(short['交易日數'].sum())}",
        f"{short['交易日數'].sum() / len(r_study):.1%}",
    ]
}, index=["全部區間段數", f"≤ {SHORT_REGIME_MONTHS} 個月的區間段數", "  └ 佔全部區間比例",
          f"  └ 其中 {LONG_OK}", f"  └ 其中 {FLAT}",
          "短區間合計交易日數", "  └ 佔研究期間比例"])
summary_short.index.name = "項目"
display(summary_short)

,值
項目,
全部區間段數,99
≤ 3 個月的區間段數,72
└ 佔全部區間比例,72.7%
└ 其中 LONG_OK,31
└ 其中 FLAT,41
短區間合計交易日數,2726
└ 佔研究期間比例,42.7%


In [23]:
short_disp = short.copy()
short_disp["起始日"] = short_disp["起始日"].dt.strftime("%Y-%m-%d")
short_disp["結束日"] = short_disp["結束日"].dt.strftime("%Y-%m-%d")
short_disp = short_disp.reset_index(drop=True)
short_disp.index += 1
display(short_disp)

record("Whipsaw 短區間檢視", True,
       f"研究期間 {len(intervals)} 段區間中，持續 ≤ {SHORT_REGIME_MONTHS} 個月者有 "
       f"{len(short)} 段（{len(short) / len(intervals):.1%}），"
       f"合計 {int(short['交易日數'].sum())} 個交易日（佔研究期間 "
       f"{short['交易日數'].sum() / len(r_study):.1%}）。僅呈現，不作處理")

,起始日,結束日,狀態,交易日數,月數
1,2001-02-01,2001-04-30,LONG_OK,61,3
2,2002-11-01,2002-12-31,LONG_OK,43,2
3,2003-01-02,2003-01-28,FLAT,19,1
4,2003-02-06,2003-02-27,LONG_OK,16,1
5,2003-03-03,2003-05-30,FLAT,64,3
6,2003-12-01,2004-01-30,FLAT,38,2
7,2004-02-02,2004-03-31,LONG_OK,43,2
8,2004-09-01,2004-10-29,LONG_OK,41,2
9,2004-11-01,2004-12-31,FLAT,45,2
10,2005-01-03,2005-03-31,LONG_OK,57,3


===> [通過] Whipsaw 短區間檢視
      研究期間 99 段區間中，持續 ≤ 3 個月者有 72 段（72.7%），合計 2726 個交易日（佔研究期間 42.7%）。僅呈現，不作處理


---
## 9. 視覺化

### 圖 1｜全期間總覽

對數座標的收盤價（2000-01-01 起），灰色半透明色塊為 `FLAT` 區間，留白處為 `LONG_OK`。
紅線為 MA50，虛線為 IS/OOS 分界（2018-01-01）。

In [24]:
def shade_flat(ax, seg_regime, color="grey", alpha=0.22):
    """在 ax 上把 FLAT 區間畫成背景色塊。"""
    iv = regime_intervals(seg_regime)
    for _, r in iv[iv["狀態"] == FLAT].iterrows():
        # 色塊延伸到下一個交易日，避免區間之間出現白縫
        loc = df.index.get_loc(r["結束日"])
        end = df.index[min(loc + 1, len(df.index) - 1)]
        ax.axvspan(r["起始日"], end, color=color, alpha=alpha, lw=0)

seg = df.loc[STUDY_START:]
fig, ax = plt.subplots(figsize=(15, 7))
ax.plot(seg.index, seg["Close"], lw=0.9, color="#3b6ea5", label="Close")
ax.plot(seg.index, seg["MA"], lw=1.0, color="#d1495b", label=f"MA{MA_LEN}")
shade_flat(ax, r_study)
ax.axvline(pd.Timestamp(OOS_START), color="black", ls="--", lw=1.2)
ax.text(pd.Timestamp(OOS_START), seg["Close"].max(), "  IS | OOS",
        va="top", ha="left", fontsize=10)
ax.set_yscale("log")
ax.set_title("TAIEX with MA50 monthly regime filter — shaded = FLAT (out of market)")
ax.set_xlabel("Date"); ax.set_ylabel("Close (log scale)")
ax.legend(loc="upper left"); ax.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

C:\Users\king5\AppData\Local\Temp\ipykernel_28836\3356488462.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### 圖 2–7｜六段重大事件放大圖

每張圖顯示該事件前後區間的收盤價與 MA50，灰色色塊為 `FLAT`，
綠色虛線為重新進場日，紅色虛線為退場日。

In [25]:
for (name, hi_win, lo_win, plot_win), (_, r) in zip(EVENTS, events_tbl.iterrows()):
    seg = df.loc[plot_win[0]:plot_win[1]]
    seg_reg = regime.loc[seg.index].dropna()

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(seg.index, seg["Close"], lw=1.1, color="#3b6ea5", label="Close")
    ax.plot(seg.index, seg["MA"], lw=1.3, color="#d1495b", label=f"MA{MA_LEN}")
    shade_flat(ax, seg_reg)

    if r["退場日"] != "—":
        ex = pd.Timestamp(r["退場日"])
        ax.axvline(ex, color="#b3261e", ls="--", lw=1.6)
        ax.annotate(f"EXIT\n{ex:%Y-%m-%d}\n{r['退場開盤']:,.0f}",
                    xy=(ex, r["退場開盤"]), xytext=(8, 14),
                    textcoords="offset points", color="#b3261e", fontsize=9,
                    arrowprops=dict(arrowstyle="->", color="#b3261e"))
    for col, pcol, lbl, dy in [
            ("進場日（退場後首次）", "進場開盤",  "RE-ENTRY\n(1st after exit)",   -34),
            ("進場日（低點後首次）", "進場開盤2", "RE-ENTRY\n(1st after trough)", -78)]:
        if r[col] == "—":
            continue
        # 兩者相同時只畫一次
        if col.endswith("低點後首次）") and r[col] == r["進場日（退場後首次）"]:
            continue
        en = pd.Timestamp(r[col])
        ax.axvline(en, color="#2e7d32", ls="--", lw=1.6)
        ax.annotate(f"{lbl}\n{en:%Y-%m-%d}\n{r[pcol]:,.0f}",
                    xy=(en, r[pcol]), xytext=(8, dy),
                    textcoords="offset points", color="#2e7d32", fontsize=9,
                    arrowprops=dict(arrowstyle="->", color="#2e7d32"))

    hi_d = pd.Timestamp(r["波段高點日"]); lo_d = pd.Timestamp(r["波段低點日"])
    ax.scatter([hi_d], [r["高點收盤"]], color="black", zorder=6, s=35)
    ax.scatter([lo_d], [r["低點收盤"]], color="black", zorder=6, s=35)
    ax.annotate(f"peak {r['高點收盤']:,.0f}", xy=(hi_d, r["高點收盤"]),
                xytext=(-10, 10), textcoords="offset points", fontsize=9, ha="right")
    ax.annotate(f"trough {r['低點收盤']:,.0f}", xy=(lo_d, r["低點收盤"]),
                xytext=(-10, -16), textcoords="offset points", fontsize=9, ha="right")

    ax.set_title(f"{name} — shaded = FLAT")
    ax.set_xlabel("Date"); ax.set_ylabel("Index level")
    ax.legend(loc="best"); ax.grid(alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    plt.tight_layout(); plt.show()

C:\Users\king5\AppData\Local\Temp\ipykernel_28836\4142872226.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()
C:\Users\king5\AppData\Local\Temp\ipykernel_28836\4142872226.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


C:\Users\king5\AppData\Local\Temp\ipykernel_28836\4142872226.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()
C:\Users\king5\AppData\Local\Temp\ipykernel_28836\4142872226.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


C:\Users\king5\AppData\Local\Temp\ipykernel_28836\4142872226.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()
C:\Users\king5\AppData\Local\Temp\ipykernel_28836\4142872226.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
## 10. 存檔

輸出 `data/regime.csv`：原始 OHLCV + MA/RSI3/RSI5（沿用，方便下游直接使用）
再加上三個新欄位。

**保留 1999 年資料不裁切。** 該期間 `regime` 為空值（MA50 尚未成形），屬正常。

存檔後讀回驗證一致性。

In [26]:
out = df.copy()
out["regime"] = regime
out["is_month_end"] = out.index.isin(month_end.values)
out["regime_change"] = (out["regime"] != out["regime"].shift(1)) & \
                       out["regime"].notna() & out["regime"].shift(1).notna()

display(out.loc[STUDY_START:].head(3))
display(out[out["regime_change"]].head(3)[["Close", "MA", "regime",
                                           "is_month_end", "regime_change"]])

,Open,High,Low,Close,Volume,MA,RSI3,RSI5,regime,is_month_end,regime_change
Date,,,,,,,,,,,
2000-01-04,"8,644.91","8,803.61","8,642.50","8,756.55",0,"7,793.72",99.08,95.34,LONG_OK,False,False
2000-01-05,"8,690.60","8,867.68","8,668.02","8,849.87",0,"7,817.39",99.28,96.02,LONG_OK,False,False
2000-01-06,"8,900.56","9,023.99","8,833.91","8,922.03",0,"7,842.73",99.42,96.51,LONG_OK,False,False


,Close,MA,regime,is_month_end,regime_change
Date,,,,,
1999-06-01,"7,397.62","7,374.03",FLAT,False,True
1999-07-02,"8,572.09","7,764.06",LONG_OK,False,True
1999-08-02,"7,195.94","7,914.18",FLAT,False,True


In [27]:
out.to_csv(DATA_OUT, date_format="%Y-%m-%d")

back = pd.read_csv(DATA_OUT, index_col="Date", parse_dates=True)
num_cols = ["Open", "High", "Low", "Close", "Volume", "MA", "RSI3", "RSI5"]
roundtrip_ok = (
    len(back) == len(out)
    and list(back.columns) == list(out.columns)
    and np.allclose(back[num_cols].to_numpy(), out[num_cols].to_numpy(), equal_nan=True)
    and back["regime"].fillna("NA").equals(out["regime"].fillna("NA"))
    and bool((back["is_month_end"] == out["is_month_end"]).all())
    and bool((back["regime_change"] == out["regime_change"]).all())
)

info = pd.DataFrame({
    "值": [
        DATA_OUT,
        f"{os.path.getsize(DATA_OUT):,} bytes",
        f"{len(back):,}",
        ", ".join(back.columns),
        f"{back.index.min():%Y-%m-%d} ~ {back.index.max():%Y-%m-%d}",
        f"{int((back.index < pd.Timestamp(STUDY_START)).sum()):,}",
        f"{int(back['regime'].isna().sum()):,}",
        f"{int(back['is_month_end'].sum()):,}",
        f"{int(back['regime_change'].sum()):,}",
    ]
}, index=["檔名", "大小", "筆數", "欄位", "期間", "1999 暖身期筆數（已保留）",
          "regime 空值筆數（皆在 MA50 成形前）", "is_month_end = True 筆數",
          "regime_change = True 筆數"])
info.index.name = "項目"
display(info)

record("存檔與讀回一致性", roundtrip_ok,
       f"{DATA_OUT} 已寫入 {len(back):,} 筆 × {len(back.columns)} 欄，讀回完全一致"
       if roundtrip_ok else "讀回的資料與記憶體中不一致，請停止並人工確認")

,值
項目,
檔名,data/regime.csv
大小,"1,011,648 bytes"
筆數,"6,632"
欄位,"Open, High, Low, Close, Volume, MA, RSI3, RSI5..."
期間,1999-01-05 ~ 2026-01-20
1999 暖身期筆數（已保留）,241
regime 空值筆數（皆在 MA50 成形前）,55
is_month_end = True 筆數,325
regime_change = True 筆數,104


===> [通過] 存檔與讀回一致性
      data/regime.csv 已寫入 6,632 筆 × 11 欄，讀回完全一致


---
## 11. 小結

In [28]:
summary = pd.DataFrame(CHECKS)
display(summary)

n_fail = int((summary["結果"] == "異常").sum())
print(f"\n共 {len(summary)} 項驗證，通過 {len(summary) - n_fail} 項，異常 {n_fail} 項")

,驗證項目,結果,說明
0,資料載入一致性,通過,"6,632 筆，1999-01-05 ~ 2026-01-20，欄位與階段 2 完全一致"
1,每月月底交易日唯一且完整,通過,325 個月，每月恰好一個月底交易日，無遺漏、無重複、無跳月
2,月度狀態判斷完成,通過,共 325 個月，其中 2 個月因 MA50 未成形而未判斷（1999-01 ~ 1999-...
3,日線狀態序列：兩種獨立建構一致,通過,"「月份 +1 對齊」與「生效日 ffill」兩條路徑在全部 6,632 個交易日上完全一致"
4,狀態來自上一個月底（程式化斷言）,通過,"研究期間 6,391 個交易日全部通過：每日 regime 皆等於其前一月份最後一個交易日的..."
5,切換時點與缺值檢查,通過,"全部 98 次切換皆發生在當月第一個交易日；2000-01-01 之後 6,391 個交易日..."
6,狀態切換清單,通過,研究期間共 98 次切換：49 次 FLAT→LONG_OK、49 次 LONG_OK→FL...
7,重大事件切換時點對照,通過,六段重大下跌的退場／進場時點已完整列表（高點→退場跌幅 -13.4% ~ -4.2%）；其中...
8,狀態統計,通過,研究期間 LONG_OK 61.15%／FLAT 38.85%；共 99 段區間，LONG_...
9,Whipsaw 短區間檢視,通過,研究期間 99 段區間中，持續 ≤ 3 個月者有 72 段（72.7%），合計 2726 個...



共 11 項驗證，通過 11 項，異常 0 項


In [29]:
# 把小結會引用到的數字集中印出，方便對照下方 markdown
key = pd.DataFrame({
    "值": [
        f"{len(monthly.dropna(subset=['判斷狀態']))}",
        f"{len(intervals)}",
        f"{len(switches)}",
        f"{n_entry}",
        f"{n_exit}",
        f"{(r_study == LONG_OK).mean():.2%}",
        f"{(r_study == FLAT).mean():.2%}",
        f"{(regime.loc[STUDY_START:IS_END] == FLAT).mean():.2%}",
        f"{(regime.loc[OOS_START:] == FLAT).mean():.2%}",
        f"{len(short)}（{len(short) / len(intervals):.1%}）",
        f"{events_tbl['高點→退場'].min():+.1%} ~ {events_tbl['高點→退場'].max():+.1%}",
        f"{events_tbl['低點→進場2'].min():+.1%} ~ {events_tbl['低點→進場2'].max():+.1%}",
        f"{int((events_tbl['進場日（退場後首次）'] != events_tbl['進場日（低點後首次）']).sum())} / 6",
    ]
}, index=["已判斷月份數", "狀態區間段數", "狀態切換總次數",
          f"{FLAT} → {LONG_OK} 次數（＝交易次數上限）", f"{LONG_OK} → {FLAT} 次數",
          f"{LONG_OK} 佔研究期間", f"{FLAT} 佔研究期間",
          "FLAT 佔 IS", "FLAT 佔 OOS",
          f"≤ {SHORT_REGIME_MONTHS} 個月的短區間", "六段事件 高點→退場 跌幅範圍",
          "六段事件 低點→進場 漲幅範圍（低點後首次）",
          "六段事件中「退場後首次進場」早於低點者"])
key.index.name = "關鍵數字"
display(key)

,值
關鍵數字,
已判斷月份數,323
狀態區間段數,99
狀態切換總次數,98
FLAT → LONG_OK 次數（＝交易次數上限）,49
LONG_OK → FLAT 次數,49
LONG_OK 佔研究期間,61.15%
FLAT 佔研究期間,38.85%
FLAT 佔 IS,40.21%
FLAT 佔 OOS,35.77%


### 本階段結論

以下只陳述觀察到的事實，不評價策略好壞、不建議改進方向。

**規則實作已驗證**

- 月底一律取自資料中實際存在的交易日，非日曆月底；每月恰好一個月底日，無遺漏、無重複、無跳月。
- 日線狀態序列用兩條**完全獨立**的路徑建構（月份週期 +1 對齊、生效日 ffill），
  在全部 6,632 個交易日上完全一致。
- 另有程式化斷言逐日驗證：研究期間每一天的 `regime` 都等於「其前一個月最後一個交易日」
  的 Close vs MA50 判斷結果，且判斷日確實落在前一個月。
- 所有狀態切換皆發生在當月第一個交易日，`2000-01-01` 之後無任何 `regime` 缺值。

**因此本層沒有 look-ahead**：判斷所需資訊在月底收盤已完全確定，狀態自次月首個交易日開盤起生效。

**狀態切換次數**

切換次數、`FLAT → LONG_OK` 次數（即最終策略的交易次數上限）、
以及 IS/OOS 分組的數字，見上方「關鍵數字」表與區塊 5 的完整清單。

**空手時間比例**

`FLAT` 佔研究期間的比例、以及 IS 與 OOS 分別的比例見上表。
區塊 7 另有逐年的 `LONG_OK` 天數與切換次數。

**六段重大事件的退場與進場時點**

區塊 6 的表格列出每一段的波段高點、退場日與跌幅、波段低點、進場日與漲幅，
補充表另有「高點→退場」「低點→進場」各隔了幾個交易日。
區塊 9 的六張放大圖可對照確認每次切換發生在走勢的哪個位置。

**「重新進場」有兩欄，兩者不一定相同。** 若下跌途中出現 whipsaw
（短暫回到 MA50 上方使月底判斷轉為 `LONG_OK`，隨即又跌破），
「退場後首次進場」會落在波段低點**之前**，補充表的「兩者相同」欄會標示為「★ 否」，
補充表的「低點→進場・退場後首次」也會呈現負的交易日數。
關鍵數字表列出六段中有幾段屬於這種情況。

月度判斷本質上是落後指標：退場必然發生在高點之後，進場必然發生在低點之後，
上述表格量化的就是這個落後的幅度。

**Whipsaw**

持續 3 個月以內的短區間段數、佔全部區間的比例、以及合計佔研究期間的交易日比例，
見區塊 8 的統計表與完整清單。**本階段未對此做任何處理。**

---

### 產出

- `data/regime.csv` —— OHLCV + MA/RSI3/RSI5 + `regime` / `is_month_end` / `regime_change`，
  保留 1999 暖身期，未裁切

### 下一階段

本階段完全未使用 RSI 產生任何訊號（RSI3/RSI5 僅沿用存檔）。
第二層的 RSI 回檔進場屬於階段 5。請先人工確認上述狀態切換清單與事件對照表，再進行下一階段。